# Bank & Agency Usage Forecast: Total → Division → Ward (Optima)

Forecasts weekly **bank** and **agency** worked hours from `ndr-hb-abuhb-sde.sde_workforce_optima.bank_agency_usage` for the next 6 months at three levels:
1. **Total**: all divisions combined
2. **Division**: every division as standard
3. **Ward**: every Optima org unit (`orgunitid`) with meaningful volume

The modelling code lives in [`forecasting.py`](forecasting.py) and is shared with the interactive dashboard (`streamlit run dashboard.py`).

**Data handling:** all record-level data stays in BigQuery. Only weekly aggregates per ward are pulled into memory; nothing is written to disk.
**Clear outputs before committing** (`Clear All Outputs`) so no figures are stored in the repo.

## 1. Setup

One-time install: `pip install -r requirements.txt`

One-time auth (in a terminal): `gcloud auth application-default login` then `gcloud auth application-default set-quota-project ndr-hb-abuhb-sde`

In [ ]:
import math
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import forecasting as fc

pd.set_option("display.float_format", "{:,.0f}".format)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 100)

### Parameters

In [ ]:
STAFF_GROUP = None             # e.g. "Nursing and Midwifery Registered" (None = all)
TRAIN_START = "2022-01-01"     # avoids the COVID-era disruption
FORECAST_MONTHS = 6
EXCLUDE_RECENT_WEEKS = 0       # increase if recent weeks are still being finalised
MODEL = fc.AUTO                # or force one of fc.MODELS, e.g. "Log ETS"

BACKTEST_STEP = 4              # total & division: thorough model selection
WARD_BACKTEST_STEP = 8         # wards: faster (hundreds of series)

MIN_DIVISION_WEEKLY_HOURS = 100   # smaller divisions are combined into OTHER
MIN_WARD_WEEKLY_HOURS = 20        # smaller wards are combined into OTHER within their division
WARD_DIVISIONS = None             # None = wards in every division, or e.g. ["Medicine", "Urgent Care"]

TYPES = list(fc.TYPES)            # ["Bank", "Agency"]; add fc.COMBINED for bank + agency together
EXPORT_TABLE = None               # e.g. "ndr-hb-abuhb-sde.sde_workforce_optima.bank_agency_forecast" (writes to BigQuery)

## 2. Connect with ADC and pull weekly aggregates per ward

In [ ]:
client = fc.get_client()
data = fc.fetch_data(client, TRAIN_START, STAFF_GROUP)
print(f"Connected to {client.project}: {len(data.units)} wards in {len(data.divisions)} divisions, "
      f"{len(data.weekly['Bank'])} weeks from {data.weekly['Bank'].index.min():%d %b %Y}")

## 3. Forecast window
Weeks end on **Saturday**. Training uses complete weeks up to the last Saturday before today; later rows are shifts booked but not yet worked, shown only as *already booked*.

In [ ]:
window = fc.make_window(FORECAST_MONTHS, EXCLUDE_RECENT_WEEKS)
print(f"Trained to week ending {window.cutoff:%d %b %Y}")
print(f"Forecast months: {window.months[0]} to {window.months[-1]} ({window.horizon} weeks)")
print("\nCandidate models:")
for name, m in fc.MODELS.items():
    print(f"  - {name}: {m.description}")

In [ ]:
def plot_result(r, ax, title, start="2024-01-01"):
    w = r.weekly
    r.actual.loc[start:].plot(ax=ax, color="#444", label="Actual")
    w["forecast"].plot(ax=ax, color="#1f77b4", lw=2, label=f"Forecast ({r.model})")
    if w["lower_80"].notna().all():
        ax.fill_between(w.index, w["lower_95"], w["upper_95"], alpha=0.12, color="#1f77b4", label="95%")
        ax.fill_between(w.index, w["lower_80"], w["upper_80"], alpha=0.25, color="#1f77b4", label="80%")
    w["prebooked_so_far"].plot(ax=ax, ls=":", color="#ff7f0e", label="Already booked")
    ax.set_title(title, fontsize=9); ax.set_xlabel("")

def run_level(items, staff_type, step, level, division=None):
    rows, results = [], {}
    for name, spec in items:
        series = fc.child_series(data, staff_type, spec)
        r = fc.run_forecast(series, window, name, MODEL, backtest_step=step)
        recent = series.loc[:window.cutoff].iloc[-fc.RECENT_WEEKS:].mean()
        rows.append({"level": level, "division": division or name, **fc.summary_row(name, r, recent),
                     "notes": "; ".join(r.notes)})
        results[name] = r
    return pd.DataFrame(rows), results

SUMMARY_COLS = ["recent_weekly_hours", "model", "backtest_wape_%", "forecast_hours", "lower_95", "upper_95",
                "avg_weekly_wte", "same_period_last_year", "change_vs_last_year_%"]

## 4. Total (all divisions)
Each candidate model is backtested at many past forecast origins on the same horizon; the lowest **WAPE** (total absolute error ÷ total actual hours) is chosen. Intervals are empirical, from those backtest errors.

In [ ]:
total_tables, total_results = [], {}
for t in TYPES:
    table, res = run_level([("ALL DIVISIONS", {})], t, BACKTEST_STEP, "Total")
    table["type"] = t
    total_tables.append(table)
    total_results[t] = next(iter(res.values()))
total_table = pd.concat(total_tables, ignore_index=True)

for t in TYPES:
    r = total_results[t]
    print(f"\n{t}: model = {r.model}  |  backtest WAPE % by model:")
    display(r.scores.round(1).to_frame("WAPE %").T)
    display(r.monthly)
display(total_table.set_index("type")[SUMMARY_COLS])

fig, axes = plt.subplots(len(TYPES), 1, figsize=(12, 3.5 * len(TYPES)), squeeze=False)
for ax, t in zip(axes.flat, TYPES):
    plot_result(total_results[t], ax, f"{t}: all divisions (worked hours per week)")
axes.flat[0].legend(loc="upper left", fontsize=8)
plt.tight_layout(); plt.show()

## 5. Division forecasts (standard)
Every division with a recent average ≥ `MIN_DIVISION_WEEKLY_HOURS` gets its own model; the rest are combined into `OTHER` so divisions add up to the total.

In [ ]:
div_tables, div_results = [], {}
for t in TYPES:
    items = fc.children(data, t, window.cutoff, min_weekly_hours=MIN_DIVISION_WEEKLY_HOURS)
    table, res = run_level(items, t, BACKTEST_STEP, "Division")
    table["type"] = t
    div_tables.append(table)
    div_results[t] = res
division_table = pd.concat(div_tables, ignore_index=True)

for t in TYPES:
    print(f"\n{t}: 6-month totals by division")
    display(division_table.query("type == @t").set_index("name")[SUMMARY_COLS]
            .sort_values("forecast_hours", ascending=False))
    print(f"{t}: monthly forecast hours by division")
    display(pd.DataFrame({n: r.monthly["forecast_hours"] for n, r in div_results[t].items()}).T.astype(float))

In [ ]:
for t in TYPES:
    names = list(div_results[t])
    nrows = math.ceil(len(names) / 3)
    fig, axes = plt.subplots(nrows, 3, figsize=(15, 3 * nrows), squeeze=False)
    for ax, name in zip(axes.flat, names):
        plot_result(div_results[t][name], ax, f"{name[:40]}\n({div_results[t][name].model})")
    for ax in axes.flat[len(names):]:
        ax.set_visible(False)
    fig.suptitle(f"{t}: worked hours per week by division (actual, forecast, 80/95% intervals)")
    plt.tight_layout(); plt.show()

## 6. Ward forecasts
Every ward (org unit) with a recent average ≥ `MIN_WARD_WEEKLY_HOURS` gets its own model; smaller wards are combined into `OTHER` within each division. New wards with short histories fall back to simpler models (see `notes`). This step runs hundreds of series and takes a few minutes.

In [ ]:
if WARD_DIVISIONS:
    ward_divisions = [d.strip().upper() for d in WARD_DIVISIONS]
    unknown = sorted(set(ward_divisions) - set(data.divisions))
    if unknown:
        raise ValueError(f"Unknown divisions {unknown}. Valid: {data.divisions}")
else:
    ward_divisions = data.divisions

started = time.time()
ward_tables, ward_results = [], {}
for t in TYPES:
    for d in ward_divisions:
        items = fc.children(data, t, window.cutoff, division=d, min_weekly_hours=MIN_WARD_WEEKLY_HOURS)
        if not items:
            continue
        table, res = run_level(items, t, WARD_BACKTEST_STEP, "Ward", division=d)
        table["type"] = t
        ward_tables.append(table)
        ward_results.update({(t, d, n): r for n, r in res.items()})
ward_table = pd.concat(ward_tables, ignore_index=True)
print(f"{len(ward_table)} ward series forecast in {time.time() - started:.0f}s")

In [ ]:
for t in TYPES:
    wt = ward_table.query("type == @t").set_index(["division", "name"])
    print(f"\n{t}: top 25 wards by forecast hours")
    display(wt[SUMMARY_COLS].sort_values("forecast_hours", ascending=False).head(25))
    print(f"{t}: largest forecast increases vs same period last year (wards with ≥ 1,000 hours last year)")
    display(wt[wt["same_period_last_year"] >= 1000][SUMMARY_COLS]
            .sort_values("change_vs_last_year_%", ascending=False).head(15))

## 7. Reconciliation
Each level is modelled independently, so children won't add up exactly to their parent. Large gaps highlight where lower-level trends disagree with the higher-level pattern.

In [ ]:
rows = []
for t in TYPES:
    total = total_results[t].summary["forecast_hours"]
    rows.append({"type": t, "parent": "ALL DIVISIONS", "children": "divisions", "parent_model": total,
                 "sum_of_children": division_table.query("type == @t")["forecast_hours"].sum()})
    for d in ward_divisions:
        sum_wards = ward_table.query("type == @t and division == @d")["forecast_hours"].sum()
        if sum_wards == 0:
            continue
        parent = div_results[t].get(d) or fc.run_forecast(data.series(t, division=d), window, d, MODEL, backtest_step=BACKTEST_STEP)
        rows.append({"type": t, "parent": d, "children": "wards", "parent_model": parent.summary["forecast_hours"],
                     "sum_of_children": sum_wards})
recon = pd.DataFrame(rows)
recon["difference_%"] = (recon["sum_of_children"] / recon["parent_model"] - 1) * 100
recon.set_index(["type", "parent"])

## 8. Combined monthly results & optional export
One tidy table across all levels. Set `EXPORT_TABLE` to write it to BigQuery (overwrites the table). Nothing is saved to the workspace.

In [ ]:
def monthly_rows(level, t, division, name, r):
    m = r.monthly.reset_index()
    return m.assign(level=level, type=t, division=division, name=name, model=r.model)

parts = [monthly_rows("Total", t, "ALL DIVISIONS", "ALL DIVISIONS", total_results[t]) for t in TYPES]
parts += [monthly_rows("Division", t, n, n, r) for t in TYPES for n, r in div_results[t].items()]
parts += [monthly_rows("Ward", t, d, n, r) for (t, d, n), r in ward_results.items()]
all_monthly = pd.concat(parts, ignore_index=True)[
    ["level", "type", "division", "name", "month", "model", "weeks", "forecast_hours",
     "lower_95", "lower_80", "upper_80", "upper_95", "avg_weekly_wte", "same_period_last_year", "change_vs_last_year_%"]]
print(f"{len(all_monthly):,} rows: " + ", ".join(f"{k} {v}" for k, v in all_monthly["level"].value_counts().items()))
all_monthly.head()

In [ ]:
if EXPORT_TABLE:
    from google.cloud import bigquery
    export = all_monthly.assign(
        month=pd.PeriodIndex(all_monthly["month"], freq="M").to_timestamp().date,
        run_date=pd.Timestamp.today().date(), cutoff_week=window.cutoff.date(), staff_group=STAFF_GROUP)
    export.columns = [c.replace("%", "pct") for c in export.columns]
    job = client.load_table_from_dataframe(export, EXPORT_TABLE,
                                           job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE"))
    job.result()
    print(f"Wrote {len(export):,} rows to {EXPORT_TABLE}")
else:
    print("EXPORT_TABLE not set; skipping export")

## Notes & limitations
- Forecasts assume current patterns continue; policy changes (agency controls, recruitment, ward openings/closures) are not modelled.
- Ward restructures (new `orgunitid`s) start a new history; short histories use simpler models and may have no intervals.
- Model choice is automatic per series; set `MODEL` to force one everywhere, or explore individual series in the dashboard.
- Recent weeks can be revised as timesheets are finalised; use `EXCLUDE_RECENT_WEEKS` if the latest week looks low.